# Notebook 13 — Tool Use and Task Success Eval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/13_tool_use_and_task_success_eval.ipynb)

This notebook builds a small, fully transparent benchmark for tool-using agents. We will score whether an agent chooses the right tools, fills the arguments correctly, and actually completes the task instead of only looking plausible.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Why step-level tool evaluation matters**
- Understand **Step 1 — Shared helpers**
- Understand **Step 2 — Define the tool catalog**
- Understand **Step 3 — Build a benchmark of tool-using tasks**


## What you will build

- a synthetic benchmark of customer-support tasks with oracle tool plans
- a lightweight tool environment with explicit success conditions
- several simple tool-using agents with different failure modes
- measurable metrics for tool choice, argument correctness, task success, and efficiency

## Why step-level tool evaluation matters

An agent can produce a helpful final answer while still using the wrong tools, skipping verification, or overusing expensive actions. For operational systems, we want to measure:

1. **tool selection accuracy** — did the agent pick the right actions?
2. **argument correctness** — did it pass the right parameters?
3. **task completion** — did those actions actually solve the problem?
4. **efficiency** — did it solve the task without unnecessary steps?

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Shared helpers

The benchmark uses only standard-library Python after setup. The helpers below keep tables readable and make artifact generation easy.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import json
import random
import statistics

random.seed(13)

ARTIFACT_DIR = Path("artifacts") / "notebook_13_tool_use"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def clip(text, width=72):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def normalize_value(value):
    if isinstance(value, float):
        return round(value, 2)
    if isinstance(value, int):
        return value
    if value is None:
        return None
    return str(value).strip().lower()


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2 — Define the tool catalog

Every tool has an explicit contract. That makes argument grading straightforward and prevents the benchmark from hiding behind vague descriptions.

In [ ]:
TOOL_SPECS = [
    {
        "tool": "search_docs",
        "required_args": ["query"],
        "role": "research",
        "description": "Look up policy or incident documentation before acting.",
    },
    {
        "tool": "lookup_order",
        "required_args": ["order_id"],
        "role": "verification",
        "description": "Fetch order details and confirm eligibility.",
    },
    {
        "tool": "issue_refund",
        "required_args": ["order_id", "amount"],
        "role": "transaction",
        "description": "Issue a refund after verification.",
    },
    {
        "tool": "create_ticket",
        "required_args": ["queue", "priority"],
        "role": "incident",
        "description": "Open an operational or engineering ticket.",
    },
    {
        "tool": "schedule_visit",
        "required_args": ["asset_id", "slot"],
        "role": "field_service",
        "description": "Book an on-site visit for a physical asset.",
    },
    {
        "tool": "escalate_case",
        "required_args": ["team", "reason"],
        "role": "governance",
        "description": "Send a case to a specialist team.",
    },
    {
        "tool": "send_message",
        "required_args": ["template"],
        "role": "communication",
        "description": "Send a templated customer-facing update.",
    },
]

TOOL_REQUIREMENTS = {
    row["tool"]: tuple(row["required_args"])
    for row in TOOL_SPECS
}

print_rows(TOOL_SPECS, columns=["tool", "role", "required_args", "description"])

## Step 3 — Build a benchmark of tool-using tasks

Each task includes:

- a user request
- structured context
- an oracle tool plan
- a terminal success condition
- allowed message templates

The benchmark is small enough to inspect by hand but broad enough to expose routing and argument mistakes.

In [ ]:
TASKS = [
    {
        "task_id": "billing_refund_01",
        "family": "billing",
        "request": "I was charged twice for order O-1001. Please refund the extra payment.",
        "context": {"customer_id": "C-1001", "order_id": "O-1001", "amount": 84.50},
        "doc_keywords": ["duplicate charge"],
        "ticket_queue": None,
        "priority": "medium",
        "asset_id": None,
        "slot": None,
        "escalation_team": None,
        "allowed_templates": ["refund_confirmed"],
        "oracle_plan": [
            {"tool": "lookup_order", "args": {"order_id": "O-1001"}},
            {"tool": "issue_refund", "args": {"order_id": "O-1001", "amount": 84.50}},
            {"tool": "send_message", "args": {"template": "refund_confirmed"}},
        ],
        "required_terminal": {"tool": "issue_refund", "args": {"order_id": "O-1001", "amount": 84.50}},
        "success_facts": ["order_verified", "refund_issued"],
    },
    {
        "task_id": "billing_refund_02",
        "family": "billing",
        "request": "My canceled add-on was still billed on order O-1008. Refund 19 dollars.",
        "context": {"customer_id": "C-1008", "order_id": "O-1008", "amount": 19.00},
        "doc_keywords": ["canceled add-on refund"],
        "ticket_queue": None,
        "priority": "medium",
        "asset_id": None,
        "slot": None,
        "escalation_team": None,
        "allowed_templates": ["refund_confirmed"],
        "oracle_plan": [
            {"tool": "lookup_order", "args": {"order_id": "O-1008"}},
            {"tool": "issue_refund", "args": {"order_id": "O-1008", "amount": 19.00}},
            {"tool": "send_message", "args": {"template": "refund_confirmed"}},
        ],
        "required_terminal": {"tool": "issue_refund", "args": {"order_id": "O-1008", "amount": 19.00}},
        "success_facts": ["order_verified", "refund_issued"],
    },
    {
        "task_id": "incident_reset_01",
        "family": "incident",
        "request": "The password reset link returns a 404 for every user in our workspace.",
        "context": {"customer_id": "C-2001", "incident_id": "IR-44"},
        "doc_keywords": ["password reset 404"],
        "ticket_queue": "auth_incidents",
        "priority": "high",
        "asset_id": None,
        "slot": None,
        "escalation_team": None,
        "allowed_templates": ["incident_opened"],
        "oracle_plan": [
            {"tool": "search_docs", "args": {"query": "password reset 404"}},
            {"tool": "create_ticket", "args": {"queue": "auth_incidents", "priority": "high"}},
            {"tool": "send_message", "args": {"template": "incident_opened"}},
        ],
        "required_terminal": {"tool": "create_ticket", "args": {"queue": "auth_incidents", "priority": "high"}},
        "success_facts": ["doc_reviewed", "ticket_created"],
    },
    {
        "task_id": "incident_outage_01",
        "family": "incident",
        "request": "Building 7 lost connectivity again. We need an outage ticket and a customer update.",
        "context": {"customer_id": "C-2007", "site": "Building 7"},
        "doc_keywords": ["building 7 outage"],
        "ticket_queue": "network_ops",
        "priority": "critical",
        "asset_id": None,
        "slot": None,
        "escalation_team": None,
        "allowed_templates": ["outage_update"],
        "oracle_plan": [
            {"tool": "search_docs", "args": {"query": "building 7 outage"}},
            {"tool": "create_ticket", "args": {"queue": "network_ops", "priority": "critical"}},
            {"tool": "send_message", "args": {"template": "outage_update"}},
        ],
        "required_terminal": {"tool": "create_ticket", "args": {"queue": "network_ops", "priority": "critical"}},
        "success_facts": ["doc_reviewed", "ticket_created"],
    },
    {
        "task_id": "shipping_delay_01",
        "family": "logistics",
        "request": "Order O-3002 is late and the tracking page stopped updating.",
        "context": {"customer_id": "C-3002", "order_id": "O-3002"},
        "doc_keywords": ["late shipment tracking"],
        "ticket_queue": None,
        "priority": "low",
        "asset_id": None,
        "slot": None,
        "escalation_team": None,
        "allowed_templates": ["delay_update"],
        "oracle_plan": [
            {"tool": "lookup_order", "args": {"order_id": "O-3002"}},
            {"tool": "send_message", "args": {"template": "delay_update"}},
        ],
        "required_terminal": {"tool": "send_message", "args": {"template": "delay_update"}},
        "success_facts": ["order_verified", "message_sent"],
    },
    {
        "task_id": "field_visit_01",
        "family": "field_service",
        "request": "Meter M-77 is sparking. Book the next safety visit and notify the customer.",
        "context": {"customer_id": "C-4001", "asset_id": "M-77", "slot": "2026-04-08 09:00"},
        "doc_keywords": ["meter sparking safety"],
        "ticket_queue": "field_safety",
        "priority": "urgent",
        "asset_id": "M-77",
        "slot": "2026-04-08 09:00",
        "escalation_team": None,
        "allowed_templates": ["visit_confirmed"],
        "oracle_plan": [
            {"tool": "create_ticket", "args": {"queue": "field_safety", "priority": "urgent"}},
            {"tool": "schedule_visit", "args": {"asset_id": "M-77", "slot": "2026-04-08 09:00"}},
            {"tool": "send_message", "args": {"template": "visit_confirmed"}},
        ],
        "required_terminal": {"tool": "schedule_visit", "args": {"asset_id": "M-77", "slot": "2026-04-08 09:00"}},
        "success_facts": ["ticket_created", "visit_booked"],
    },
    {
        "task_id": "appointment_reschedule_01",
        "family": "field_service",
        "request": "Please move asset A-12 maintenance to the afternoon slot.",
        "context": {"customer_id": "C-4012", "asset_id": "A-12", "slot": "2026-04-09 14:00"},
        "doc_keywords": ["appointment reschedule"],
        "ticket_queue": None,
        "priority": "low",
        "asset_id": "A-12",
        "slot": "2026-04-09 14:00",
        "escalation_team": None,
        "allowed_templates": ["visit_confirmed"],
        "oracle_plan": [
            {"tool": "schedule_visit", "args": {"asset_id": "A-12", "slot": "2026-04-09 14:00"}},
            {"tool": "send_message", "args": {"template": "visit_confirmed"}},
        ],
        "required_terminal": {"tool": "schedule_visit", "args": {"asset_id": "A-12", "slot": "2026-04-09 14:00"}},
        "success_facts": ["visit_booked"],
    },
    {
        "task_id": "privacy_export_01",
        "family": "privacy",
        "request": "I need a full data export under the privacy policy.",
        "context": {"customer_id": "C-5001", "reason": "data_export"},
        "doc_keywords": ["data export privacy"],
        "ticket_queue": None,
        "priority": "high",
        "asset_id": None,
        "slot": None,
        "escalation_team": "privacy_ops",
        "allowed_templates": ["privacy_ack"],
        "oracle_plan": [
            {"tool": "search_docs", "args": {"query": "data export privacy"}},
            {"tool": "escalate_case", "args": {"team": "privacy_ops", "reason": "data_export"}},
            {"tool": "send_message", "args": {"template": "privacy_ack"}},
        ],
        "required_terminal": {"tool": "escalate_case", "args": {"team": "privacy_ops", "reason": "data_export"}},
        "success_facts": ["doc_reviewed", "case_escalated"],
    },
]

print_rows(
    [
        {
            "task_id": task["task_id"],
            "family": task["family"],
            "request": clip(task["request"], 62),
            "oracle_tools": " -> ".join(step["tool"] for step in task["oracle_plan"]),
        }
        for task in TASKS
    ],
    columns=["task_id", "family", "request", "oracle_tools"],
)

## Step 4 — Inspect benchmark balance

Before scoring agents, verify that the benchmark covers multiple families and multiple plan lengths. This prevents a misleading benchmark that is dominated by one easy workflow.

In [ ]:
family_counts = Counter(task["family"] for task in TASKS)
plan_length_rows = [
    {
        "task_id": task["task_id"],
        "family": task["family"],
        "plan_length": len(task["oracle_plan"]),
    }
    for task in TASKS
]

print("Family counts:", dict(family_counts))
print()
print_rows(plan_length_rows, columns=["task_id", "family", "plan_length"])

## Step 5 — Create a lightweight tool environment

The environment does not try to be realistic in every detail. It only needs to make success measurable. Each tool either unlocks a fact, fails, or both.

In [ ]:
def required_args_present(tool_name, args):
    required = TOOL_REQUIREMENTS[tool_name]
    return all(key in args and args[key] not in (None, "") for key in required)


def args_match(expected_args, predicted_args):
    return all(
        normalize_value(predicted_args.get(key)) == normalize_value(value)
        for key, value in expected_args.items()
    )


def execute_tool(task, action, state):
    tool = action["tool"]
    args = action.get("args", {})
    record = {
        "tool": tool,
        "args": dict(args),
        "args_present": required_args_present(tool, args),
        "ok": False,
        "new_facts": [],
        "message": "",
    }

    if tool == "search_docs":
        expected_query = task["doc_keywords"][0]
        if record["args_present"] and expected_query in str(args.get("query", "")).lower():
            record["ok"] = True
            record["new_facts"] = ["doc_reviewed"]
            record["message"] = "Relevant documentation found."
        else:
            record["message"] = "Irrelevant or missing query."

    elif tool == "lookup_order":
        expected_order_id = task["context"].get("order_id")
        if expected_order_id and record["args_present"] and args.get("order_id") == expected_order_id:
            record["ok"] = True
            record["new_facts"] = ["order_verified"]
            record["message"] = "Order verified."
        else:
            record["message"] = "Order lookup failed."

    elif tool == "issue_refund":
        expected = task["required_terminal"]["args"]
        if "order_verified" in state["facts"] and record["args_present"] and args_match(expected, args):
            record["ok"] = True
            record["new_facts"] = ["refund_issued"]
            record["message"] = "Refund issued."
        else:
            record["message"] = "Refund rejected because verification or amount is wrong."

    elif tool == "create_ticket":
        expected = task["required_terminal"]["args"]
        allowed_exact = tool == task["required_terminal"]["tool"]
        queue_ok = args.get("queue") == task["ticket_queue"]
        priority_ok = normalize_value(args.get("priority")) == normalize_value(task["priority"])
        if record["args_present"] and queue_ok and priority_ok:
            record["ok"] = True
            record["new_facts"] = ["ticket_created"]
            record["message"] = "Ticket created."
        elif allowed_exact and args_match(expected, args):
            record["ok"] = True
            record["new_facts"] = ["ticket_created"]
            record["message"] = "Ticket created."
        else:
            record["message"] = "Ticket created in the wrong queue or with the wrong priority."

    elif tool == "schedule_visit":
        expected = task["required_terminal"]["args"]
        if record["args_present"] and args_match(expected, args):
            record["ok"] = True
            record["new_facts"] = ["visit_booked"]
            record["message"] = "Visit scheduled."
        else:
            record["message"] = "Schedule request does not match the required slot."

    elif tool == "escalate_case":
        expected = task["required_terminal"]["args"]
        if record["args_present"] and args_match(expected, args):
            record["ok"] = True
            record["new_facts"] = ["case_escalated"]
            record["message"] = "Case escalated."
        else:
            record["message"] = "Escalated to the wrong team or with the wrong reason."

    elif tool == "send_message":
        template = args.get("template")
        if record["args_present"] and template in task["allowed_templates"]:
            record["ok"] = True
            record["new_facts"] = ["message_sent"]
            record["message"] = "Message sent."
        else:
            record["message"] = "Unknown template."

    state["facts"].update(record["new_facts"])
    state["history"].append(record)
    return record


def run_actions(task, actions):
    state = {"facts": set(), "history": []}
    for action in actions:
        execute_tool(task, action, state)
    terminal = task["required_terminal"]
    terminal_ok = any(
        step["tool"] == terminal["tool"] and args_match(terminal["args"], step["args"]) and step["ok"]
        for step in state["history"]
    )
    success = terminal_ok and set(task["success_facts"]).issubset(state["facts"])
    return {
        "success": success,
        "facts": sorted(state["facts"]),
        "history": state["history"],
    }

## Step 6 — Implement three simple agents

The agents are intentionally lightweight. They are not meant to be strong models; they are meant to expose different evaluation signals:

- **careful_agent** usually succeeds but sometimes adds redundant checks
- **guessy_agent** skips verification and guesses arguments
- **chatty_agent** overuses tools and occasionally escalates unnecessarily

In [ ]:
import re
import json as _json


def action(tool, **args):
    return {"tool": tool, "args": dict(args)}


TOOL_DESCRIPTIONS = "\n".join(
    f"- {spec['tool']}({', '.join(spec['required_args'])}): {spec['description']}"
    for spec in TOOL_SPECS
)


def _build_task_context(task):
    """Format task fields into a prompt string for the model."""
    parts = [f"Customer request: {task['request']}"]
    ctx = task["context"]
    for key, value in ctx.items():
        parts.append(f"  {key}: {value}")
    if task.get("ticket_queue"):
        parts.append(f"  ticket_queue: {task['ticket_queue']}")
    if task.get("priority"):
        parts.append(f"  priority: {task['priority']}")
    if task.get("asset_id"):
        parts.append(f"  asset_id: {task['asset_id']}")
    if task.get("slot"):
        parts.append(f"  slot: {task['slot']}")
    if task.get("escalation_team"):
        parts.append(f"  escalation_team: {task['escalation_team']}")
    if task.get("allowed_templates"):
        parts.append(f"  allowed_templates: {task['allowed_templates']}")
    if task.get("doc_keywords"):
        parts.append(f"  doc_keywords: {task['doc_keywords']}")
    return "\n".join(parts)


def parse_tool_calls(raw_text):
    """Extract a list of action dicts from model output.

    Tries three strategies in order:
    1. Find a JSON array containing objects with a "tool" key.
    2. Find individual JSON objects with a "tool" key.
    3. Match function-call-style patterns like tool_name(arg=val).
    """
    actions = []

    # Strategy 1: JSON array
    for m in re.finditer(r'\[[\s\S]*?\{[\s\S]*?"tool"[\s\S]*?\]', raw_text):
        try:
            candidates = _json.loads(m.group())
        except _json.JSONDecodeError:
            continue
        if isinstance(candidates, list):
            for item in candidates:
                if isinstance(item, dict) and "tool" in item:
                    args = item.get("args", {})
                    if not isinstance(args, dict):
                        args = {}
                    actions.append({"tool": item["tool"], "args": args})
            if actions:
                return actions

    # Strategy 2: individual JSON objects
    for m in re.finditer(r'\{[^{}]*"tool"\s*:\s*"[^"]+?"[^{}]*\}', raw_text):
        try:
            item = _json.loads(m.group())
        except _json.JSONDecodeError:
            continue
        if "tool" in item:
            actions.append({"tool": item["tool"], "args": item.get("args", {})})
    if actions:
        return actions

    # Strategy 3: function-call patterns
    valid_tools = {spec["tool"] for spec in TOOL_SPECS}
    for m in re.finditer(r'(\w+)\(([^)]*)\)', raw_text):
        tool_name = m.group(1)
        if tool_name not in valid_tools:
            continue
        arg_str = m.group(2)
        args = {}
        for am in re.finditer(r'(\w+)\s*=\s*["\']?([^"\'`,]+)["\']?', arg_str):
            key, val = am.group(1), am.group(2).strip()
            try:
                val = float(val) if "." in val else int(val)
            except ValueError:
                pass
            args[key] = val
        actions.append({"tool": tool_name, "args": args})
    return actions


_TOOL_CALL_FORMAT = (
    'Output ONLY a JSON array of tool calls. Each element must have '
    '"tool" (string) and "args" (object) keys.\n'
    'Example: '
    '[{"tool": "lookup_order", "args": {"order_id": "O-1234"}}, '
    '{"tool": "send_message", "args": {"template": "refund_confirmed"}}]'
)


def careful_agent(task):
    """Careful agent: detailed prompt, greedy decoding, plans before acting."""
    context_str = _build_task_context(task)
    system_prompt = (
        "You are a careful customer-support agent. Available tools:\n"
        + TOOL_DESCRIPTIONS + "\n\n"
        "Instructions:\n"
        "1. Read the customer request and all context carefully.\n"
        "2. Always verify information before taking action "
        "(e.g., look up an order before issuing a refund).\n"
        "3. Search documentation when relevant keywords are present.\n"
        "4. After completing the main action, send an appropriate "
        "message to the customer using the allowed template.\n"
        "5. " + _TOOL_CALL_FORMAT
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context_str},
    ]
    raw = generate(messages, max_new_tokens=512, temperature=0.0, do_sample=False)
    return parse_tool_calls(raw) or []


def guessy_agent(task):
    """Guessy agent: minimal prompt, higher temperature, skips verification."""
    context_str = _build_task_context(task)
    system_prompt = (
        "You are a support agent. Tools:\n"
        + TOOL_DESCRIPTIONS + "\n\n"
        + _TOOL_CALL_FORMAT + "\n"
        "Be quick, act immediately."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context_str},
    ]
    raw = generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True)
    return parse_tool_calls(raw) or []


def chatty_agent(task):
    """Chatty agent: verbose prompt encourages explanation over action."""
    context_str = _build_task_context(task)
    system_prompt = (
        "You are a friendly, thorough customer-support agent who likes "
        "to explain your reasoning in detail. Available tools:\n"
        + TOOL_DESCRIPTIONS + "\n\n"
        "For each step, explain WHY you are calling that tool. Search "
        "documentation for background first. Look up information from "
        "multiple angles. If anything seems unusual, escalate to a "
        "specialist team.\n\n"
        "After your explanation, output your tool calls as a JSON array "
        'where each element has "tool" and "args" keys.'
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context_str},
    ]
    raw = generate(messages, max_new_tokens=768, temperature=0.7, do_sample=True)
    return parse_tool_calls(raw) or []


AGENTS = {
    "careful_agent": careful_agent,
    "guessy_agent": guessy_agent,
    "chatty_agent": chatty_agent,
}


## Step 7 — Inspect one task by hand

Before benchmarking everything, inspect a single task and confirm that the environment behaves the way you expect.

In [ ]:
demo_task = next(task for task in TASKS if task["task_id"] == "billing_refund_01")
for agent_name, agent_fn in AGENTS.items():
    predicted_actions = agent_fn(demo_task)
    execution = run_actions(demo_task, predicted_actions)
    print("=" * 72)
    print(agent_name)
    print("Actions:", predicted_actions)
    print("Facts:", execution["facts"])
    print("Success:", execution["success"])

## Step 8 — Score tool selection and argument correctness

We will use a greedy sequence alignment so agents can receive partial credit for getting some tools right even when they add extra actions.

In [ ]:
def align_actions(oracle_actions, predicted_actions):
    matches = []
    start = 0
    for oracle_index, oracle_step in enumerate(oracle_actions):
        for predicted_index in range(start, len(predicted_actions)):
            predicted_step = predicted_actions[predicted_index]
            if predicted_step["tool"] == oracle_step["tool"]:
                matches.append((oracle_index, predicted_index, oracle_step, predicted_step))
                start = predicted_index + 1
                break
    return matches


def compare_args(expected_args, predicted_args):
    if not expected_args:
        return 1.0
    hits = 0
    for key, value in expected_args.items():
        if normalize_value(predicted_args.get(key)) == normalize_value(value):
            hits += 1
    return hits / len(expected_args)


def score_agent_run(task, agent_name, predicted_actions, execution):
    oracle_actions = task["oracle_plan"]
    matches = align_actions(oracle_actions, predicted_actions)

    per_expected_step = {index: {"matched": False, "arg_score": 0.0} for index in range(len(oracle_actions))}
    for oracle_index, predicted_index, oracle_step, predicted_step in matches:
        per_expected_step[oracle_index] = {
            "matched": True,
            "arg_score": compare_args(oracle_step["args"], predicted_step["args"]),
            "predicted_index": predicted_index,
        }

    tool_recall = len(matches) / len(oracle_actions)
    tool_precision = len(matches) / len(predicted_actions) if predicted_actions else 0.0
    first_tool_correct = int(
        bool(predicted_actions)
        and predicted_actions[0]["tool"] == oracle_actions[0]["tool"]
    )
    sequence_exact = int([step["tool"] for step in predicted_actions] == [step["tool"] for step in oracle_actions])
    argument_accuracy = statistics.fmean(
        item["arg_score"] for item in per_expected_step.values()
    )
    efficiency = min(1.0, len(oracle_actions) / len(predicted_actions)) if predicted_actions else 0.0
    overall = round(
        0.35 * tool_recall
        + 0.20 * tool_precision
        + 0.20 * argument_accuracy
        + 0.20 * int(execution["success"])
        + 0.05 * efficiency,
        3,
    )

    summary = {
        "agent": agent_name,
        "task_id": task["task_id"],
        "family": task["family"],
        "tool_recall": round(tool_recall, 3),
        "tool_precision": round(tool_precision, 3),
        "argument_accuracy": round(argument_accuracy, 3),
        "first_tool_correct": first_tool_correct,
        "sequence_exact": sequence_exact,
        "task_success": int(execution["success"]),
        "efficiency": round(efficiency, 3),
        "overall_score": overall,
        "expected_first_tool": oracle_actions[0]["tool"],
        "predicted_first_tool": predicted_actions[0]["tool"] if predicted_actions else "(none)",
        "predicted_steps": len(predicted_actions),
        "oracle_steps": len(oracle_actions),
        "facts": ", ".join(execution["facts"]),
    }

    step_records = []
    for expected_index, oracle_step in enumerate(oracle_actions):
        item = per_expected_step[expected_index]
        step_records.append(
            {
                "agent": agent_name,
                "task_id": task["task_id"],
                "family": task["family"],
                "expected_step": expected_index + 1,
                "expected_tool": oracle_step["tool"],
                "matched": int(item["matched"]),
                "arg_score": round(item["arg_score"], 3),
                "predicted_index": item.get("predicted_index", -1),
            }
        )

    for predicted_index, predicted_step in enumerate(predicted_actions):
        if predicted_index not in {match[1] for match in matches}:
            step_records.append(
                {
                    "agent": agent_name,
                    "task_id": task["task_id"],
                    "family": task["family"],
                    "expected_step": None,
                    "expected_tool": "(extra)",
                    "matched": 0,
                    "arg_score": 0.0,
                    "predicted_index": predicted_index,
                    "extra_tool": predicted_step["tool"],
                }
            )

    return summary, step_records

## Step 9 — Run the benchmark

Now we can execute every task for every agent and collect task-level as well as step-level records.

In [ ]:
task_level_results = []
step_level_records = []

for task in TASKS:
    for agent_name, agent_fn in AGENTS.items():
        predicted_actions = agent_fn(task)
        execution = run_actions(task, predicted_actions)
        summary, steps = score_agent_run(task, agent_name, predicted_actions, execution)
        task_level_results.append(summary)
        step_level_records.extend(steps)

print("Task runs:", len(task_level_results))
print("Step records:", len(step_level_records))

## Step 10 — Build the leaderboard

The main benchmark table combines tool correctness and end-to-end success. This makes it easy to see whether an agent is merely chatty or actually reliable.

In [ ]:
leaderboard = []
for agent_name in AGENTS:
    rows = [row for row in task_level_results if row["agent"] == agent_name]
    leaderboard.append(
        {
            "agent": agent_name,
            "overall_score": round(statistics.fmean(row["overall_score"] for row in rows), 3),
            "tool_recall": round(statistics.fmean(row["tool_recall"] for row in rows), 3),
            "tool_precision": round(statistics.fmean(row["tool_precision"] for row in rows), 3),
            "argument_accuracy": round(statistics.fmean(row["argument_accuracy"] for row in rows), 3),
            "first_tool_accuracy": round(statistics.fmean(row["first_tool_correct"] for row in rows), 3),
            "sequence_exact_rate": round(statistics.fmean(row["sequence_exact"] for row in rows), 3),
            "task_success_rate": round(statistics.fmean(row["task_success"] for row in rows), 3),
            "avg_steps": round(statistics.fmean(row["predicted_steps"] for row in rows), 2),
        }
    )

leaderboard = sorted(leaderboard, key=lambda row: row["overall_score"], reverse=True)
print_rows(
    leaderboard,
    columns=[
        "agent",
        "overall_score",
        "tool_recall",
        "tool_precision",
        "argument_accuracy",
        "first_tool_accuracy",
        "sequence_exact_rate",
        "task_success_rate",
        "avg_steps",
    ],
)

## Step 11 — Compare performance by task family

Family-level slices show where agents are strong or weak. Some agents may look fine overall but collapse on privacy or incident workflows.

In [ ]:
family_summary = []
for agent_name in AGENTS:
    for family in sorted({task["family"] for task in TASKS}):
        rows = [
            row for row in task_level_results
            if row["agent"] == agent_name and row["family"] == family
        ]
        family_summary.append(
            {
                "agent": agent_name,
                "family": family,
                "overall_score": round(statistics.fmean(row["overall_score"] for row in rows), 3),
                "task_success_rate": round(statistics.fmean(row["task_success"] for row in rows), 3),
                "tool_recall": round(statistics.fmean(row["tool_recall"] for row in rows), 3),
                "argument_accuracy": round(statistics.fmean(row["argument_accuracy"] for row in rows), 3),
            }
        )

print_rows(
    family_summary,
    columns=["agent", "family", "overall_score", "task_success_rate", "tool_recall", "argument_accuracy"],
)

## Step 12 — Measure first-tool confusion

Tool-using agents often fail at the very first decision. A first-tool confusion table is a cheap way to spot routing bias.

In [ ]:
confusion = defaultdict(Counter)
for row in task_level_results:
    confusion[row["expected_first_tool"]][row["predicted_first_tool"]] += 1

confusion_rows = []
for expected_tool, counter in sorted(confusion.items()):
    for predicted_tool, count in counter.items():
        confusion_rows.append(
            {
                "expected_first_tool": expected_tool,
                "predicted_first_tool": predicted_tool,
                "count": count,
            }
        )

print_rows(confusion_rows, columns=["expected_first_tool", "predicted_first_tool", "count"])

## Step 13 — Bucket the argument failures

Sequence accuracy alone is not enough. We also want to know whether failures came from missing arguments, wrong values, or extra actions.

In [ ]:
failure_buckets = Counter()
for row in step_level_records:
    if row["expected_tool"] == "(extra)":
        failure_buckets["extra_tool"] += 1
    elif row["matched"] == 0:
        failure_buckets["missing_required_tool"] += 1
    elif row["arg_score"] < 1.0:
        failure_buckets["bad_arguments"] += 1
    else:
        failure_buckets["clean_step"] += 1

bucket_rows = [
    {"bucket": bucket, "count": count}
    for bucket, count in sorted(failure_buckets.items())
]
print_rows(bucket_rows, columns=["bucket", "count"])

## Step 14 — Inspect efficiency versus task success

Over-tooling is often hidden by binary success metrics. Here we compare average step counts with the task completion rate.

In [ ]:
efficiency_rows = []
for row in leaderboard:
    efficiency_rows.append(
        {
            "agent": row["agent"],
            "avg_steps": row["avg_steps"],
            "task_success_rate": row["task_success_rate"],
            "efficiency_gap": round(row["avg_steps"] - statistics.fmean(task["oracle_steps"] for task in [item for item in task_level_results if item["agent"] == row["agent"]]), 3),
        }
    )

print_rows(efficiency_rows, columns=["agent", "avg_steps", "task_success_rate", "efficiency_gap"])

## Step 15 — Save measurable artifacts

The notebook should leave behind structured outputs that can be reused later in a regression suite or CI workflow.

In [ ]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


(ARTIFACT_DIR / "task_level_results.json").write_text(
    json.dumps(task_level_results, indent=2),
    encoding="utf-8",
)
(ARTIFACT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2),
    encoding="utf-8",
)
(ARTIFACT_DIR / "family_summary.json").write_text(
    json.dumps(family_summary, indent=2),
    encoding="utf-8",
)
write_csv(ARTIFACT_DIR / "step_level_records.csv", step_level_records)
write_csv(ARTIFACT_DIR / "task_level_results.csv", task_level_results)

print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name)

## Step 16 — Produce an operator summary

End with a compact report that explains what the metrics mean and where the next round of improvements should focus.

In [ ]:
best_agent = leaderboard[0]["agent"]
worst_agent = leaderboard[-1]["agent"]

recommendations = []
if any(row["task_success_rate"] < 0.8 for row in leaderboard):
    recommendations.append("Add regression gates on task success before promoting agent changes.")
if any(row["argument_accuracy"] < 0.9 for row in leaderboard):
    recommendations.append("Introduce stricter argument templates or schema validation for tool calls.")
if any(row["tool_precision"] < 0.8 for row in leaderboard):
    recommendations.append("Penalize unnecessary tools to reduce cost and confusion.")

report = {
    "best_agent": best_agent,
    "worst_agent": worst_agent,
    "top_overall_score": leaderboard[0]["overall_score"],
    "top_task_success_rate": leaderboard[0]["task_success_rate"],
    "largest_failure_bucket": bucket_rows[0] if bucket_rows else None,
    "recommendations": recommendations,
}

report_path = ARTIFACT_DIR / "operator_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

## Recap

You now have a first-principles tool-use evaluation workflow with:

- explicit tasks and oracle tool plans
- argument-aware tool scoring
- task completion metrics tied to environment state
- reusable JSON and CSV artifacts

This is the core pattern for evaluating simple tool-using agents before moving on to trajectory grading and multi-agent systems.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [12 Rag Ablation Lab](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/12_rag_ablation_lab.ipynb) | ➡️ **Next:** [14 Trajectory Grading](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/14_trajectory_grading.ipynb)